<a href="https://colab.research.google.com/github/naman39910/voyage-analytics/blob/main/nootbooks%20/hypothesis_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **VOYAGE ANALYTICS — HYPOTHESIS TESTING**
## Goal : Statistically validate EDA findings
## Tests : ANOVA, Chi-Square, Kruskal-Wallis, T-Test

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Statistical tests
from scipy import stats
from scipy.stats import (
    f_oneway,        # One-way ANOVA
    kruskal,         # Kruskal-Wallis (non-parametric ANOVA)
    chi2_contingency,# Chi-Square test
    ttest_ind,       # Independent T-test
    mannwhitneyu,    # Mann-Whitney U (non-parametric t-test)
    shapiro          # Normality test
)

ALPHA = 0.05  # significance level

print("-> All libraries imported")
print(f"Significance Level (α) = {ALPHA}")

## Load and Merge Data

In [ ]:
users   = pd.read_csv('/content/users.csv')
flights = pd.read_csv('/content/flights.csv')
hotels  = pd.read_csv('/content/hotels.csv')

# Merge users with flights
users_flights = flights.merge(
    users[['code', 'gender', 'age', 'company']],
    left_on='userCode', right_on='code', how='left'
)

# Merge users with hotels
users_hotels = hotels.merge(
    users[['code', 'gender', 'age', 'company']],
    left_on='userCode', right_on='code', how='left'
)

print("Users + Flights shape :", users_flights.shape)
print("Users + Hotels shape :", users_hotels.shape)

# Gender groups for easy access
genders = ['male', 'female', 'none']

print("Gender values:", users['gender'].unique())

male_users = users[users['gender'] == 'male']
female_users = users[users['gender'] == 'female']
none_users = users[users['gender'] == 'none']

print(f"\nGender counts:")
print(f" Male : {len(male_users)}")
print(f" Female : {len(female_users)}")
print(f" None : {len(none_users)}")

## Resources

### Helper Function for Hypothesis Testing

In [ ]:
def conduct_hypothesis_test(p_value, alpha=ALPHA):
    """
    Conducts a hypothesis test based on the p-value and significance level.
    Prints whether to reject or fail to reject the null hypothesis.

    Args:
        p_value (float): The p-value obtained from the statistical test.
        alpha (float, optional): The significance level. Defaults to ALPHA (0.05).
    """
    print(f"P-value: {p_value:.4f}")
    print(f"Significance Level (α): {alpha}")

    if p_value < alpha:
        print("Conclusion: Reject the Null Hypothesis (H0).")
    else:
        print("Conclusion: Fail to Reject the Null Hypothesis (H0).")

## Hypothesis 1: Age across Gender Groups

**H0 (Null Hypothesis)**: There is no significant difference in the mean age across different gender groups.

**H1 (Alternative Hypothesis)**: There is a significant difference in the mean age across different gender groups.

### Assumptions for ANOVA / Kruskal-Wallis

Before conducting the test, we need to check the assumptions:

1.  **Independence**: The samples are independent (different gender groups).
2.  **Normality**: The data for each group should be approximately normally distributed. (If not, use Kruskal-Wallis).
3.  **Homogeneity of Variances**: The variances of the groups should be approximately equal. (If not, use Welch's ANOVA or Kruskal-Wallis).

Let's start by checking the normality assumption for each gender group's age distribution using the Shapiro-Wilk test.

In [ ]:
# Check for normality using Shapiro-Wilk test
# H0: The data is normally distributed
# H1: The data is not normally distributed

print("Shapiro-Wilk Test for Normality (Age by Gender):")
for gender in users['gender'].unique():
    group_ages = users[users['gender'] == gender]['age']
    stat, p = shapiro(group_ages)
    print(f"  {gender.capitalize()} group (N={len(group_ages)}):")
    print(f"    Statistics={stat:.3f}, p={p:.3f}")
    if p > ALPHA:
        print('    Sample looks Gaussian (fail to reject H0)')
    else:
        print('    Sample does not look Gaussian (reject H0)')

# Visualize age distribution for each gender
plt.figure(figsize=(10, 6))
sns.histplot(data=users, x='age', hue='gender', kde=True, palette='viridis')
plt.title('Age Distribution by Gender')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

### Test for Homogeneity of Variances (Levene's Test)

Although we've decided to use Kruskal-Wallis due to non-normality, it's good practice to check for homogeneity of variances for completeness or if a different non-parametric test might be considered.

**H0 (Null Hypothesis)**: The variances of the age groups are equal.

**H1 (Alternative Hypothesis)**: The variances of the age groups are not equal.

In [ ]:
# Prepare data for Levene's test
male_ages = users[users['gender'] == 'male']['age']
female_ages = users[users['gender'] == 'female']['age']
none_ages = users[users['gender'] == 'none']['age']

# Perform Levene's test
stat, p_levene = stats.levene(male_ages, female_ages, none_ages)

print(f"Levene's Test for Homogeneity of Variances (Age by Gender):")
print(f"  Statistics={stat:.3f}, p={p_levene:.3f}")

if p_levene > ALPHA:
    print('  Variances are equal (fail to reject H0)')
else:
    print('  Variances are not equal (reject H0)')

### Perform Kruskal-Wallis H-test

Since the normality assumption was violated for all groups, we will use the non-parametric Kruskal-Wallis H-test to compare the medians of age across the different gender groups.

In [ ]:
# Perform Kruskal-Wallis H-test
stat_kruskal, p_kruskal = kruskal(male_ages, female_ages, none_ages)

print("Kruskal-Wallis H-test for Age across Gender Groups:")
print(f"  Statistics={stat_kruskal:.3f}")

conduct_hypothesis_test(p_kruskal)

### Conclusion for Hypothesis 1

Based on the Kruskal-Wallis H-test, with a p-value of 0.6161 (which is greater than our significance level α = 0.05), we **fail to reject the null hypothesis**. This suggests that there is no statistically significant difference in the median age across the different gender groups in this dataset.

## Hypothesis 2: Gender Distribution across Companies

**H0 (Null Hypothesis)**: There is no significant association between gender and company.

**H1 (Alternative Hypothesis)**: There is a significant association between gender and company.

### Test for Association using Chi-Square Test

To test for an association between two categorical variables (gender and company), we will use the Chi-Square test of independence.

**Assumptions for Chi-Square Test**:

1.  **Independence**: Observations are independent.
2.  **Expected Frequencies**: Expected frequencies for each cell in the contingency table should be at least 5 (though some allow for fewer, ideally >1).

Let's create a contingency table and perform the Chi-Square test.

In [ ]:
# Create a contingency table of Gender vs. Company
contingency_table = pd.crosstab(users['gender'], users['company'])

print("Contingency Table (Gender vs. Company):")
print(contingency_table)

# Perform Chi-Square Test of Independence
chi2, p_chi2, dof, expected = chi2_contingency(contingency_table)

print(f"\nChi-Square Statistic: {chi2:.3f}")
print(f"Degrees of Freedom: {dof}")
print("Expected Frequencies Table:")
print(pd.DataFrame(expected, index=contingency_table.index, columns=contingency_table.columns))

conduct_hypothesis_test(p_chi2)

### Conclusion for Hypothesis 1

Based on the Kruskal-Wallis H-test, with a p-value of 0.6161 (which is greater than our significance level α = 0.05), we **fail to reject the null hypothesis**. This suggests that there is no statistically significant difference in the median age across the different gender groups in this dataset.

## Hypothesis 3: Age vs. Travel Preferences (Flights)

**H0 (Null Hypothesis)**: There is no significant correlation between a user's age and the price of flights they book.

**H1 (Alternative Hypothesis)**: There is a significant correlation between a user's age and the price of flights they book.

### Test for Correlation (Pearson / Spearman)

To test the relationship between two continuous variables (age and flight price), we can use correlation tests. We should first check for normality and visualize the relationship.

**Assumptions for Pearson Correlation**:

1.  **Linearity**: There is a linear relationship between the two variables.
2.  **Normality**: Both variables are approximately normally distributed.
3.  **Homoscedasticity**: The variance of the residuals is constant across all levels of the independent variable.

If these assumptions are not met, we can use the non-parametric Spearman correlation.

Let's first visualize the relationship and check for normality of the flight prices.

In [ ]:
# Merge age into the flights DataFrame
flights_with_age = flights.merge(users[['code', 'age']], left_on='userCode', right_on='code', how='left')

# Drop rows where 'age' or 'price' might be missing after merge (if any)
flights_with_age.dropna(subset=['age', 'price'], inplace=True)

print(f"Shape of flights_with_age: {flights_with_age.shape}")
print(f"Unique ages: {flights_with_age['age'].nunique()}")
print(f"Unique flight prices: {flights_with_age['price'].nunique()}")

# Visualize the relationship between Age and Flight Price
plt.figure(figsize=(10, 6))
sns.scatterplot(data=flights_with_age, x='age', y='price', alpha=0.5)
plt.title('Age vs. Flight Price')
plt.xlabel('Age')
plt.ylabel('Flight Price')
plt.show()

# Check normality for flight prices (and age, though already checked for users['age'])
stat_price, p_price = shapiro(flights_with_age['price'])
print(f"\nShapiro-Wilk Test for Normality (Flight Price):")
print(f"  Statistics={stat_price:.3f}, p={p_price:.3f}")
if p_price > ALPHA:
    print('  Flight Price sample looks Gaussian (fail to reject H0)')
else:
    print('  Flight Price sample does not look Gaussian (reject H0)')

### Conclusion for Hypothesis 2

Based on the Chi-Square test of independence, with a p-value of 0.3750 (which is greater than our significance level α = 0.05), we **fail to reject the null hypothesis**. This suggests that there is no statistically significant association between gender and company in this dataset. The distribution of genders across companies appears to be independent.

## Hypothesis 4: Gender and Hotel Preferences

**H0 (Null Hypothesis)**: There is no significant association between a user's gender and the number of days they book hotels for.

**H1 (Alternative Hypothesis)**: There is a significant association between a user's gender and the number of days they book hotels for.

### Test for Association (Chi-Square Test)

We will use the Chi-Square test of independence to examine the association between gender (categorical) and the number of hotel booking days (categorical, if treated as discrete categories).

Let's create a contingency table and then perform the Chi-Square test.

In [ ]:
# Create a contingency table of Gender vs. Hotel Days
contingency_table_hotels = pd.crosstab(users_hotels['gender'], users_hotels['days'])

print("Contingency Table (Gender vs. Hotel Days):")
print(contingency_table_hotels)

# Perform Chi-Square Test of Independence
chi2_hotels, p_chi2_hotels, dof_hotels, expected_hotels = chi2_contingency(contingency_table_hotels)

print(f"\nChi-Square Statistic: {chi2_hotels:.3f}")
print(f"Degrees of Freedom: {dof_hotels}")
print("Expected Frequencies Table:")
print(pd.DataFrame(expected_hotels, index=contingency_table_hotels.index, columns=contingency_table_hotels.columns))

conduct_hypothesis_test(p_chi2_hotels)

### Conclusion for Hypothesis 3

Based on the Spearman correlation test, with a p-value of 0.0000 (which is less than our significance level α = 0.05), we **reject the null hypothesis**. This suggests that there is a statistically significant correlation between a user's age and the price of flights they book. However, the Spearman correlation coefficient is -0.011, indicating a very weak, almost negligible, negative monotonic relationship. This means that while statistically significant, the practical impact of age on flight price is minimal in this dataset.

## Hypothesis 5: Hotel Prices across Gender Groups

**H0 (Null Hypothesis)**: There is no significant difference in the mean hotel prices across different gender groups.

**H1 (Alternative Hypothesis)**: There is a significant difference in the mean hotel prices across different gender groups.

### Assumptions for ANOVA / Kruskal-Wallis

Before conducting the test, we need to check the assumptions for comparing means/medians across groups:

1.  **Independence**: The samples are independent (different gender groups).
2.  **Normality**: The hotel prices for each group should be approximately normally distributed. (If not, use Kruskal-Wallis).
3.  **Homogeneity of Variances**: The variances of the groups should be approximately equal. (If not, use Welch's ANOVA or Kruskal-Wallis).

Let's start by checking the normality assumption for each gender group's hotel price distribution using the Shapiro-Wilk test.

In [ ]:
# Check for normality using Shapiro-Wilk test
# H0: The data is normally distributed
# H1: The data is not normally distributed

print("Shapiro-Wilk Test for Normality (Hotel Price by Gender):")
for gender in users_hotels['gender'].unique():
    group_prices = users_hotels[users_hotels['gender'] == gender]['price']
    if len(group_prices) > 1: # Shapiro-Wilk requires at least 2 data points
        stat, p = shapiro(group_prices)
        print(f"  {gender.capitalize()} group (N={len(group_prices)}):")
        print(f"    Statistics={stat:.3f}, p={p:.3f}")
        if p > ALPHA:
            print('    Sample looks Gaussian (fail to reject H0)')
        else:
            print('    Sample does not look Gaussian (reject H0)')
    else:
        print(f"  {gender.capitalize()} group (N={len(group_prices)}): Not enough data for Shapiro-Wilk test")

# Visualize hotel price distribution for each gender
plt.figure(figsize=(10, 6))
sns.histplot(data=users_hotels, x='price', hue='gender', kde=True, palette='viridis')
plt.title('Hotel Price Distribution by Gender')
plt.xlabel('Hotel Price')
plt.ylabel('Frequency')
plt.show()

### Conclusion for Hypothesis 4

Based on the Chi-Square test of independence, with a p-value of 0.2427 (which is greater than our significance level α = 0.05), we **fail to reject the null hypothesis**. This suggests that there is no statistically significant association between a user's gender and the number of days they book hotels for in this dataset.

## Hypothesis 6: Flight Prices across Gender Groups

**H0 (Null Hypothesis)**: There is no significant difference in the mean flight prices across different gender groups.

**H1 (Alternative Hypothesis)**: There is a significant difference in the mean flight prices across different gender groups.

### Assumptions for ANOVA / Kruskal-Wallis

Before conducting the test, we need to check the assumptions for comparing means/medians across groups:

1.  **Independence**: The samples are independent (different gender groups).
2.  **Normality**: The flight prices for each group should be approximately normally distributed. (If not, use Kruskal-Wallis).
3.  **Homogeneity of Variances**: The variances of the groups should be approximately equal. (If not, use Welch's ANOVA or Kruskal-Wallis).

From previous checks, we observed that flight prices are not normally distributed across gender groups. Therefore, we will proceed with checking the homogeneity of variances and then use the Kruskal-Wallis H-test.

### Test for Homogeneity of Variances (Levene's Test)

Although we've observed non-normality, it's good practice to check for homogeneity of variances for completeness or if a different non-parametric test might be considered.

**H0 (Null Hypothesis)**: The variances of the flight prices across gender groups are equal.

**H1 (Alternative Hypothesis)**: The variances of the flight prices across gender groups are not equal.

In [ ]:
# Prepare data for Levene's test for flight prices
male_flight_prices = users_flights[users_flights['gender'] == 'male']['price']
female_flight_prices = users_flights[users_flights['gender'] == 'female']['price']
none_flight_prices = users_flights[users_flights['gender'] == 'none']['price']

# Perform Levene's test
stat_levene_flight, p_levene_flight = stats.levene(male_flight_prices, female_flight_prices, none_flight_prices)

print(f"Levene's Test for Homogeneity of Variances (Flight Price by Gender):")
print(f"  Statistics={stat_levene_flight:.3f}, p={p_levene_flight:.3f}")

if p_levene_flight > ALPHA:
    print('  Variances are equal (fail to reject H0)')
else:
    print('  Variances are not equal (reject H0)')

### Perform Kruskal-Wallis H-test

Since the normality assumption was violated for all groups, we will use the non-parametric Kruskal-Wallis H-test to compare the medians of flight prices across the different gender groups.

In [ ]:
# Perform Kruskal-Wallis H-test
stat_kruskal_flight, p_kruskal_flight = kruskal(male_flight_prices, female_flight_prices, none_flight_prices)

print("Kruskal-Wallis H-test for Flight Prices across Gender Groups:")
print(f"  Statistics={stat_kruskal_flight:.3f}")

conduct_hypothesis_test(p_kruskal_flight)

### Conclusion for Hypothesis 6

Based on the Kruskal-Wallis H-test, with a p-value of 0.0018 (which is less than our significance level α = 0.05), we **reject the null hypothesis**. This suggests that there is a statistically significant difference in the median flight prices across the different gender groups in this dataset. The Levene's test indicated that variances were not homogeneous (p=0.000), which further supported the use of a non-parametric test like Kruskal-Wallis.